# Analyze Garmin data prepocess it & export to blender for 3D vizualization

## Sources:
- [Analysis of Running Activities from Garmin Watch Using Python](https://towardsdatascience.com/analysis-of-runing-activities-from-garmin-watch-using-python-99609f83314e)

## Import modules

In [1]:
# import internal modules
# from typing import List, Set, Dict, TypedDict, Tuple, Optional, Union
from pathlib import Path

# import 3rd-party modules
import pandas as pd

# import local modules

## Define functions

In [145]:
# Create Function to explore the dataframes
def explore(df: pd.DataFrame) -> None:
    """
    Function to print general information about the dataframe
    """
    print("********** 1. General info of data **********")
    print(df.info())
    
    print("\n********** 2. Shape of data **********")
    print(f"Number of rows: {len(df)}")
    print(f"Number of columns: {len(df.columns)}")
    
    print("\n********** 3. Number of missing values per column **********")
    print(df.isnull().sum())
    
    print("\n********** 4. Number of duplicated values **********")
    print(df.duplicated().sum())
    
    print("\n********** 5. Number of unique values per column (NaN non included) **********")
    print(df.nunique())
    
    print("\n********** 6. Statistical info of each column **********")
    # print(df.describe(include='all').T)
    return df.describe(include='all', datetime_is_numeric=True).T

def convert_strings_to_duration(str_series):
    """
    Function to convert series of strings to durations; workaround when multiple formats in series
    """
    return pd.to_timedelta(pd.to_datetime(str_series).dt.strftime("%H:%M:%S.%f"))

def convert_durations_to_minutes(durations_series):
    # durations_series.dt.hour*60 + durations_series.dt.minute + durations_series.dt.second/60
    return durations_series.dt.total_seconds()/60

## Read data

In [137]:
# set csv path
football_activities_df_path = Path("assets/data/garmin_data/football_1_year.csv")

# read csv into dataframe
football_activities_df = pd.read_csv(football_activities_df_path, parse_dates=True)

## Explore data (Exploratory data analysis)

In [10]:
football_activities_df.head()

,Activity Type,Date,Favorite,Title,Distance,Calories,Time,Avg HR,Max HR,Aerobic TE,...,Min Temp,Surface Interval,Decompression,Best Lap Time,Number of Laps,Max Temp,Moving Time,Elapsed Time,Min Elevation,Max Elevation
0,Other,2022-08-07 13:49:30,False,Grimbergen Football,8.90,"1,381",02:45:10,140,192,5.0,...,0.0,0:00,No,11:37.86.7,9,0.0,01:49:59,02:45:10,40,42
1,Other,2022-07-31 16:41:23,False,Grimbergen Football,0.62,375,00:30:13,152,185,3.3,...,0.0,0:00,No,30:13.30.7,1,0.0,00:13:03,00:30:13,40,42
2,Other,2022-07-31 14:20:23,False,Grimbergen Football,6.71,775,01:39:29,129,172,3.0,...,0.0,0:00,No,12:10.61.4,7,0.0,01:22:37,01:39:30,40,44
3,Other,2022-07-24 14:43:01,False,Grimbergen Football,4.96,854,01:32:10,137,180,3.4,...,0.0,0:00,No,11:03.70.5,5,0.0,01:07:09,01:49:56,40,45
4,Other,2022-07-17 13:53:02,False,Grimbergen Football,9.09,"1,479",02:22:50,150,195,5.0,...,0.0,0:00,No,04:52.04.1,10,0.0,01:52:40,02:23:16,40,45


In [11]:
explore(football_activities_df)

********** 1. General info of data **********
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 41 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Activity Type             40 non-null     object 
 1   Date                      40 non-null     object 
 2   Favorite                  40 non-null     bool   
 3   Title                     40 non-null     object 
 4   Distance                  40 non-null     float64
 5   Calories                  40 non-null     object 
 6   Time                      40 non-null     object 
 7   Avg HR                    40 non-null     int64  
 8   Max HR                    40 non-null     int64  
 9   Aerobic TE                40 non-null     object 
 10  Avg Run Cadence           40 non-null     object 
 11  Max Run Cadence           40 non-null     object 
 12  Avg Speed                 40 non-null     object 
 13  Max Speed            

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Activity Type,40,1,Other,40,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Date,40,40,2022-07-10 14:36:44,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Favorite,40,1,False,40,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Title,40,5,Grimbergen Football,33,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Distance,40.0,NaN,NaN,NaN,4.45575,3.036394,0.0,1.65,4.72,6.6275,10.14
Calories,40,40,260,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Time,40,40,00:30:13,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Avg HR,40.0,NaN,NaN,NaN,139.125,13.291963,113.0,130.75,140.5,150.5,161.0
Max HR,40.0,NaN,NaN,NaN,173.65,20.041687,134.0,161.75,181.0,190.5,199.0
Aerobic TE,40,24,5.0,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Clean data

In [138]:
# select columns to keep
selected_cols = [
    "Date",
    "Title",
    "Distance",
    "Calories",	"Time", "Avg HR", "Max HR", "Aerobic TE", "Avg Run Cadence",
    "Max Run Cadence", "Avg Speed", "Max Speed", "Total Ascent", "Total Descent", "Avg Stride Length", 
    "Best Lap Time", "Number of Laps", "Max Temp", "Moving Time", "Elapsed Time", "Min Elevation", "Max Elevation"
    ]

football_activities_df = football_activities_df[selected_cols]

### ToDo: find bettr solution to parse duration strings
ideas: 
- coerce error to convert error to nan and then fill na with other format
- consolidate strings in 1 format


In [139]:
# convert concerned cols to datetime & duration time
football_activities_df['Date'] = pd.to_datetime(football_activities_df['Date'])
football_activities_df['Time'] = convert_strings_to_duration(football_activities_df['Time'])
football_activities_df['Elapsed Time'] = convert_strings_to_duration(football_activities_df['Elapsed Time'])
football_activities_df['Best Lap Time'] = football_activities_df['Best Lap Time'].apply(lambda x: f"00:{x}" if x.count(":") == 1 else x) # need to add hh:
football_activities_df['Best Lap Time'] = pd.to_timedelta(football_activities_df['Best Lap Time'])
football_activities_df['Moving Time'] = convert_strings_to_duration(football_activities_df['Moving Time'])

In [146]:
# convert datetime cols to number of minutes
football_activities_df['Time'] = convert_durations_to_minutes(football_activities_df['Time'])
football_activities_df['Elapsed Time'] = convert_durations_to_minutes(football_activities_df['Elapsed Time'])
football_activities_df['Best Lap Time'] = convert_durations_to_minutes(football_activities_df['Best Lap Time'])
football_activities_df['Moving Time'] = convert_durations_to_minutes(football_activities_df['Moving Time'])

In [147]:
football_activities_df[["Time", 'Moving Time', "Elapsed Time", "Best Lap Time"]].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Time           40 non-null     float64
 1   Moving Time    40 non-null     float64
 2   Elapsed Time   40 non-null     float64
 3   Best Lap Time  40 non-null     float64
dtypes: float64(4)
memory usage: 1.4 KB


In [148]:
football_activities_df[["Time", 'Moving Time', "Elapsed Time", "Best Lap Time"]]

,Time,Moving Time,Elapsed Time,Best Lap Time
0,165.166667,109.983333,165.166667,11.631117
1,30.216667,13.050000,30.216667,30.221783
2,99.483333,82.616667,99.500000,12.176900
3,92.166667,67.150000,109.933333,11.061750
4,142.833333,112.666667,143.266667,4.867350
5,60.550000,50.800000,60.550000,14.148567
6,123.733333,89.916667,123.816667,4.826667
7,46.616667,35.950000,46.616667,0.061567
8,63.800000,52.633333,63.800000,9.890883
9,40.033333,30.133333,40.033333,11.074117


In [64]:
football_activities_df["Time"]

0    2022-08-08 02:45:10.000
1    2022-08-08 00:30:13.000
2    2022-08-08 01:39:29.000
3    2022-08-08 01:32:10.000
4    2022-08-08 02:22:50.000
5    2022-08-08 01:00:33.000
6    2022-08-08 02:03:44.000
7    2022-08-08 00:46:37.000
8    2022-08-08 01:03:48.000
9    2022-08-08 00:40:02.000
10   2022-08-08 01:03:13.000
11   2022-08-08 00:41:31.000
12   2022-08-08 01:42:09.000
13   2022-08-08 02:23:19.000
14   2022-08-08 01:02:01.000
15   2022-08-08 00:15:46.000
16   2022-08-08 01:28:06.000
17   2022-08-08 00:38:43.000
18   2022-08-08 02:00:22.000
19   2022-08-08 01:19:29.000
20   2022-08-08 01:37:53.000
21   2022-08-08 00:56:42.000
22   2022-08-08 00:09:14.700
23   2022-08-08 01:24:41.000
24   2022-08-08 01:15:45.000
25   2022-08-08 00:54:19.000
26   2022-08-08 00:06:15.000
27   2022-08-08 00:23:46.000
28   2022-08-08 00:10:58.000
29   2022-08-08 01:04:21.000
30   2022-08-08 01:54:58.000
31   2022-08-08 01:12:59.000
32   2022-08-08 00:03:35.200
33   2022-08-08 01:18:06.000
34   2022-08-0

In [46]:
football_activities_df[["Max Speed"]]

,Max Speed
0,22.3
1,15.5
2,23.6
3,19.9
4,23.6
5,22.9
6,20.5
7,19.7
8,19.4
9,17.7


In [ ]:


#convert 'Avg Pace', 'Best Pace', 'Elapced Time' objects to the number of minutes
df['Avg Pace'] = df['Avg Pace'].dt.hour*60 + df['Avg Pace'].dt.minute + df['Avg Pace'].dt.second/60
df['Best Pace'] = df['Best Pace'].dt.hour*60 + df['Best Pace'].dt.minute + df['Best Pace'].dt.second/60
df['Elapsed Time'] = df['Elapsed Time'].dt.hour*60 + df['Elapsed Time'].dt.minute + df['Elapsed Time'].dt.second/60